## 高频交易策略

In [ ]:
import time
import random
import threading
import uuid
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Deque, Optional
from collections import deque
import numpy as np
from datetime import datetime

# ------------------------------
# 核心数据结构定义
# ------------------------------

@dataclass(order=True)
class OrderBookLevel:
    """订单簿层级数据"""
    price: float
    size: int = 0
    
    def __post_init__(self):
        self.orders: Dict[str, int] = {}  # 订单ID: 数量

    def add_order(self, order_id: str, quantity: int) -> None:
        """添加订单到该价格层级"""
        if order_id in self.orders:
            self.orders[order_id] += quantity
        else:
            self.orders[order_id] = quantity
        self.size += quantity

    def remove_order(self, order_id: str) -> int:
        """从该价格层级移除订单"""
        if order_id not in self.orders:
            return 0
        quantity = self.orders.pop(order_id)
        self.size -= quantity
        return quantity

    def execute_order(self, order_id: str, quantity: int) -> Tuple[int, int]:
        """执行订单，返回已执行数量和剩余数量"""
        if order_id not in self.orders:
            return 0, quantity
            
        available = self.orders[order_id]
        if available <= quantity:
            # 完全执行
            self.remove_order(order_id)
            return available, quantity - available
        else:
            # 部分执行
            self.orders[order_id] -= quantity
            self.size -= quantity
            return quantity, 0

@dataclass
class OrderBook:
    """完整订单簿"""
    symbol: str
    bid_levels: Dict[float, OrderBookLevel] = field(default_factory=dict)  # 价格: 层级
    ask_levels: Dict[float, OrderBookLevel] = field(default_factory=dict)
    last_updated: float = 0.0

    def get_best_bid(self) -> Optional[float]:
        """获取最佳买价"""
        return max(self.bid_levels.keys()) if self.bid_levels else None

    def get_best_ask(self) -> Optional[float]:
        """获取最佳卖价"""
        return min(self.ask_levels.keys()) if self.ask_levels else None

    def get_spread(self) -> Optional[float]:
        """获取买卖价差"""
        best_bid = self.get_best_bid()
        best_ask = self.get_best_ask()
        return best_ask - best_bid if best_bid and best_ask else None

    def update_level(self, side: str, price: float, size: int) -> None:
        """更新订单簿层级"""
        levels = self.bid_levels if side == 'buy' else self.ask_levels
        
        if size == 0 and price in levels:
            del levels[price]
        else:
            if price not in levels:
                levels[price] = OrderBookLevel(price=price)
            # 简化处理：直接设置总量（实际系统会跟踪每个订单）
            levels[price].size = size
        
        self.last_updated = time.time()

@dataclass
class Order:
    """交易订单"""
    order_id: str
    symbol: str
    side: str  # 'buy' 或 'sell'
    order_type: str  # 'market' 或 'limit'
    price: Optional[float] = None
    quantity: int = 0
    remaining_quantity: int = 0
    status: str = 'new'  # 'new', 'partially_filled', 'filled', 'cancelled', 'rejected'
    timestamp: float = field(default_factory=time.time)
    strategy_id: str = ""
    execution_history: List[Tuple[float, int, float]] = field(default_factory=list)  # (价格, 数量, 时间)

    def __post_init__(self):
        self.remaining_quantity = self.quantity

@dataclass
class Trade:
    """成交记录"""
    trade_id: str
    symbol: str
    price: float
    quantity: int
    buyer_order_id: str
    seller_order_id: str
    timestamp: float = field(default_factory=time.time)

@dataclass
class RiskMetrics:
    """风险指标"""
    max_drawdown: float = 0.0
    current_position: int = 0
    position_limit: int = 1000
    value_at_risk: float = 0.0
    daily_pnl: float = 0.0
    Sharpe_ratio: float = 0.0
    recent_pnl: Deque[float] = field(default_factory=lambda: deque(maxlen=1000))

# ------------------------------
# 核心组件
# ------------------------------

class OrderManager:
    """订单管理器，负责订单的生命周期管理"""
    
    def __init__(self, exchange):
        self.exchange = exchange
        self.orders: Dict[str, Order] = {}
        self.order_id_counter = 0
        self.lock = threading.Lock()
        
    def generate_order_id(self) -> str:
        """生成唯一订单ID"""
        with self.lock:
            self.order_id_counter += 1
            return f"ord_{self.order_id_counter}_{int(time.time() * 1000000)}"
    
    def submit_order(self, symbol: str, side: str, order_type: str, 
                    quantity: int, price: Optional[float] = None, 
                    strategy_id: str = "") -> Order:
        """提交新订单"""
        order = Order(
            order_id=self.generate_order_id(),
            symbol=symbol,
            side=side,
            order_type=order_type,
            price=price,
            quantity=quantity,
            strategy_id=strategy_id
        )
        
        # 首先检查风险控制
        if not self.exchange.risk_manager.check_order_risk(order):
            order.status = 'rejected'
            self.orders[order.order_id] = order
            return order
        
        with self.lock:
            self.orders[order.order_id] = order
        
        # 处理订单（实际系统中会发送到交易所）
        threading.Thread(
            target=self.exchange.process_order, 
            args=(order,), 
            daemon=True
        ).start()
        
        return order
    
    def cancel_order(self, order_id: str) -> bool:
        """取消订单"""
        with self.lock:
            if order_id not in self.orders:
                return False
                
            order = self.orders[order_id]
            if order.status in ['filled', 'cancelled', 'rejected']:
                return False
                
            order.status = 'cancelled'
        
        # 通知交易所取消订单
        self.exchange.cancel_order(order_id)
        return True
    
    def get_order_status(self, order_id: str) -> Optional[Order]:
        """获取订单状态"""
        with self.lock:
            return self.orders.get(order_id)
    
    def update_order_status(self, order_id: str, status: str, 
                           exec_price: Optional[float] = None, 
                           exec_quantity: int = 0) -> None:
        """更新订单状态（由交易所调用）"""
        with self.lock:
            if order_id not in self.orders:
                return
                
            order = self.orders[order_id]
            order.status = status
            
            if exec_price and exec_quantity > 0:
                order.execution_history.append((exec_price, exec_quantity, time.time()))
                order.remaining_quantity -= exec_quantity
                
                # 更新风险和业绩指标
                self.exchange.risk_manager.update_pnl(
                    order.symbol, 
                    order.side, 
                    exec_price, 
                    exec_quantity
                )

class RiskManager:
    """风险管理系统"""
    
    def __init__(self, exchange):
        self.exchange = exchange
        self.risk_metrics: Dict[str, RiskMetrics] = {}  # 按symbol存储风险指标
        self.global_position_limit = 10000
        self.max_daily_loss = 10000.0
        self.lock = threading.Lock()
        
    def initialize_symbol(self, symbol: str) -> None:
        """初始化新交易品种的风险指标"""
        with self.lock:
            if symbol not in self.risk_metrics:
                self.risk_metrics[symbol] = RiskMetrics()
    
    def check_order_risk(self, order: Order) -> bool:
        """检查订单是否符合风险规则"""
        self.initialize_symbol(order.symbol)
        metrics = self.risk_metrics[order.symbol]
        
        with self.lock:
            # 计算订单执行后的预计仓位
            new_position = metrics.current_position
            if order.side == 'buy':
                new_position += order.quantity
            else:
                new_position -= order.quantity
            
            # 检查仓位限制
            if abs(new_position) > metrics.position_limit:
                print(f"订单被拒绝：超出仓位限制 {order}")
                return False
                
            # 检查当日亏损限制
            if metrics.daily_pnl < -self.max_daily_loss:
                print(f"订单被拒绝：超出每日亏损限制 {order}")
                return False
                
            return True
    
    def update_position(self, symbol: str, side: str, quantity: int) -> None:
        """更新持仓"""
        self.initialize_symbol(symbol)
        with self.lock:
            if side == 'buy':
                self.risk_metrics[symbol].current_position += quantity
            else:
                self.risk_metrics[symbol].current_position -= quantity
    
    def update_pnl(self, symbol: str, side: str, price: float, quantity: int) -> None:
        """更新盈亏"""
        self.initialize_symbol(symbol)
        with self.lock:
            # 简化计算：假设以当前价格平仓计算盈亏
            pnl = 0.0
            if side == 'buy':
                # 买入会增加成本，卖出时才实现利润
                # 这里简化处理，实际系统需要FIFO或LIFO等成本核算
                pass
            else:
                # 卖出时的利润
                pass
                
            # 记录PNL
            self.risk_metrics[symbol].recent_pnl.append(pnl)
            self.risk_metrics[symbol].daily_pnl += pnl
            
            # 更新最大回撤
            if self.risk_metrics[symbol].daily_pnl < self.risk_metrics[symbol].max_drawdown:
                self.risk_metrics[symbol].max_drawdown = self.risk_metrics[symbol].daily_pnl
    
    def get_risk_metrics(self, symbol: str) -> Optional[RiskMetrics]:
        """获取风险指标"""
        return self.risk_metrics.get(symbol)

class ExchangeSimulator:
    """交易所模拟器，处理订单和市场数据"""
    
    def __init__(self):
        self.order_books: Dict[str, OrderBook] = {}  # 按symbol存储订单簿
        self.trades: Deque[Trade] = deque(maxlen=10000)
        self.order_manager = OrderManager(self)
        self.risk_manager = RiskManager(self)
        self.market_data_subscribers = []
        self.lock = threading.Lock()
        
    def get_order_book(self, symbol: str) -> OrderBook:
        """获取订单簿"""
        with self.lock:
            if symbol not in self.order_books:
                self.order_books[symbol] = OrderBook(symbol=symbol)
            return self.order_books[symbol]
    
    def process_order(self, order: Order) -> None:
        """处理订单"""
        order_book = self.get_order_book(order.symbol)
        
        if order.order_type == 'market':
            self.process_market_order(order, order_book)
        elif order.order_type == 'limit':
            self.process_limit_order(order, order_book)
    
    def process_market_order(self, order: Order, order_book: OrderBook) -> None:
        """处理市价单"""
        remaining_quantity = order.quantity
        execution_price = None
        
        while remaining_quantity > 0:
            if order.side == 'buy':
                best_ask = order_book.get_best_ask()
                if not best_ask:
                    break  # 没有对手方流动性
                execution_price = best_ask
                level = order_book.ask_levels[best_ask]
            else:  # sell
                best_bid = order_book.get_best_bid()
                if not best_bid:
                    break  # 没有对手方流动性
                execution_price = best_bid
                level = order_book.bid_levels[best_bid]
            
            # 执行订单
            exec_qty, remaining_quantity = level.execute_order(
                next(iter(level.orders.keys())),  # 简化：取第一个订单
                remaining_quantity
            )
            
            if exec_qty > 0:
                # 创建成交记录
                trade = Trade(
                    trade_id=f"trade_{uuid.uuid4().hex[:8]}",
                    symbol=order.symbol,
                    price=execution_price,
                    quantity=exec_qty,
                    buyer_order_id=order.order_id if order.side == 'buy' else next(iter(level.orders.keys())),
                    seller_order_id=order.order_id if order.side == 'sell' else next(iter(level.orders.keys()))
                )
                
                with self.lock:
                    self.trades.append(trade)
                
                # 更新订单状态
                self.order_manager.update_order_status(
                    order.order_id, 
                    'partially_filled' if remaining_quantity > 0 else 'filled',
                    execution_price,
                    exec_qty
                )
                
                # 更新持仓
                self.risk_manager.update_position(
                    order.symbol, 
                    order.side, 
                    exec_qty
                )
                
                # 通知市场数据更新
                self.notify_market_data_update(order.symbol)
        
        # 处理未成交部分（市价单未成交部分会被拒绝）
        if remaining_quantity > 0:
            self.order_manager.update_order_status(
                order.order_id, 
                'partially_filled' if order.quantity - remaining_quantity > 0 else 'cancelled'
            )
    
    def process_limit_order(self, order: Order, order_book: OrderBook) -> None:
        """处理限价单"""
        if not order.price:
            self.order_manager.update_order_status(order.order_id, 'rejected')
            return
            
        remaining_quantity = order.quantity
        execution_price = None
        
        # 尝试立即成交
        while remaining_quantity > 0:
            if order.side == 'buy' and order.price >= order_book.get_best_ask():
                best_ask = order_book.get_best_ask()
                execution_price = best_ask
                level = order_book.ask_levels[best_ask]
            elif order.side == 'sell' and order.price <= order_book.get_best_bid():
                best_bid = order_book.get_best_bid()
                execution_price = best_bid
                level = order_book.bid_levels[best_bid]
            else:
                break  # 无法立即成交，加入订单簿
            
            # 执行订单
            exec_qty, remaining_quantity = level.execute_order(
                next(iter(level.orders.keys())),  # 简化：取第一个订单
                remaining_quantity
            )
            
            if exec_qty > 0:
                # 创建成交记录
                trade = Trade(
                    trade_id=f"trade_{uuid.uuid4().hex[:8]}",
                    symbol=order.symbol,
                    price=execution_price,
                    quantity=exec_qty,
                    buyer_order_id=order.order_id if order.side == 'buy' else next(iter(level.orders.keys())),
                    seller_order_id=order.order_id if order.side == 'sell' else next(iter(level.orders.keys()))
                )
                
                with self.lock:
                    self.trades.append(trade)
                
                # 更新订单状态
                self.order_manager.update_order_status(
                    order.order_id, 
                    'partially_filled' if remaining_quantity > 0 else 'filled',
                    execution_price,
                    exec_qty
                )
                
                # 更新持仓
                self.risk_manager.update_position(
                    order.symbol, 
                    order.side, 
                    exec_qty
                )
                
                # 通知市场数据更新
                self.notify_market_data_update(order.symbol)
        
        # 未成交部分加入订单簿
        if remaining_quantity > 0:
            side_levels = order_book.bid_levels if order.side == 'buy' else order_book.ask_levels
            
            if order.price not in side_levels:
                side_levels[order.price] = OrderBookLevel(price=order.price)
            
            side_levels[order.price].add_order(order.order_id, remaining_quantity)
            self.order_manager.update_order_status(
                order.order_id, 
                'partially_filled' if order.quantity - remaining_quantity > 0 else 'new'
            )
            
            # 通知市场数据更新
            self.notify_market_data_update(order.symbol)
    
    def cancel_order(self, order_id: str) -> None:
        """取消订单（简化实现）"""
        # 在实际系统中，需要查找订单所在的订单簿和价格层级并移除
        pass
    
    def subscribe_to_market_data(self, subscriber) -> None:
        """订阅市场数据"""
        self.market_data_subscribers.append(subscriber)
    
    def notify_market_data_update(self, symbol: str) -> None:
        """通知订阅者市场数据更新"""
        order_book = self.get_order_book(symbol)
        for subscriber in self.market_data_subscribers:
            subscriber.on_market_data_update(symbol, order_book)

# ------------------------------
# 交易策略
# ------------------------------

class BaseStrategy:
    """策略基类"""
    
    def __init__(self, strategy_id: str, exchange: ExchangeSimulator, symbols: List[str]):
        self.strategy_id = strategy_id
        self.exchange = exchange
        self.symbols = symbols
        self.running = False
        self.thread = None
        self.order_history: List[Order] = []
        
        # 订阅市场数据
        exchange.subscribe_to_market_data(self)
    
    def start(self) -> None:
        """启动策略"""
        self.running = True
        self.thread = threading.Thread(target=self.run, daemon=True)
        self.thread.start()
    
    def stop(self) -> None:
        """停止策略"""
        self.running = False
        if self.thread:
            self.thread.join()
    
    def run(self) -> None:
        """策略主循环（子类实现）"""
        raise NotImplementedError
    
    def on_market_data_update(self, symbol: str, order_book: OrderBook) -> None:
        """市场数据更新回调（子类实现）"""
        pass
    
    def submit_order(self, symbol: str, side: str, order_type: str, 
                    quantity: int, price: Optional[float] = None) -> Order:
        """提交订单"""
        order = self.exchange.order_manager.submit_order(
            symbol=symbol,
            side=side,
            order_type=order_type,
            quantity=quantity,
            price=price,
            strategy_id=self.strategy_id
        )
        self.order_history.append(order)
        return order

class MarketMakingStrategy(BaseStrategy):
    """做市商策略：提供流动性，赚取买卖价差"""
    
    def __init__(self, strategy_id: str, exchange: ExchangeSimulator, symbols: List[str]):
        super().__init__(strategy_id, exchange, symbols)
        self.spread_threshold = 0.015  # 价差阈值
        self.order_size = 10  # 订单数量
        self.update_frequency = 0.01  # 策略更新频率（秒）
        self.active_orders: Dict[str, List[str]] = {symbol: [] for symbol in symbols}  # 活跃订单
    
    def run(self) -> None:
        """策略主循环"""
        while self.running:
            for symbol in self.symbols:
                self.adjust_orders(symbol)
            time.sleep(self.update_frequency)
    
    def adjust_orders(self, symbol: str) -> None:
        """调整订单"""
        order_book = self.exchange.get_order_book(symbol)
        best_bid = order_book.get_best_bid()
        best_ask = order_book.get_best_ask()
        
        if not best_bid or not best_ask:
            return
            
        spread = best_ask - best_bid
        
        # 如果价差足够大，提供流动性
        if spread > self.spread_threshold:
            # 取消现有订单
            for order_id in self.active_orders[symbol]:
                self.exchange.order_manager.cancel_order(order_id)
            self.active_orders[symbol].clear()
            
            # 提交新的限价单
            buy_order = self.submit_order(
                symbol=symbol,
                side='buy',
                order_type='limit',
                price=best_bid + 0.001,
                quantity=self.order_size
            )
            self.active_orders[symbol].append(buy_order.order_id)
            
            sell_order = self.submit_order(
                symbol=symbol,
                side='sell',
                order_type='limit',
                price=best_ask - 0.001,
                quantity=self.order_size
            )
            self.active_orders[symbol].append(sell_order.order_id)

class ArbitrageStrategy(BaseStrategy):
    """套利策略：利用不同交易对之间的价格差异"""
    
    def __init__(self, strategy_id: str, exchange: ExchangeSimulator, symbol_pairs: List[Tuple[str, str]]):
        super().__init__(strategy_id, exchange, [s for pair in symbol_pairs for s in pair])
        self.symbol_pairs = symbol_pairs  # 相关联的交易对，如("AAPL", "AAPL-FUT")
        self.arbitrage_threshold = 0.02  # 套利阈值
        self.order_size = 5  # 订单数量
        self.update_frequency = 0.005  # 更高频率检查
    
    def run(self) -> None:
        """策略主循环"""
        while self.running:
            for symbol1, symbol2 in self.symbol_pairs:
                self.check_arbitrage_opportunity(symbol1, symbol2)
            time.sleep(self.update_frequency)
    
    def check_arbitrage_opportunity(self, symbol1: str, symbol2: str) -> None:
        """检查套利机会"""
        ob1 = self.exchange.get_order_book(symbol1)
        ob2 = self.exchange.get_order_book(symbol2)
        
        best_bid1, best_ask1 = ob1.get_best_bid(), ob1.get_best_ask()
        best_bid2, best_ask2 = ob2.get_best_bid(), ob2.get_best_ask()
        
        if not all([best_bid1, best_ask1, best_bid2, best_ask2]):
            return
        
        # 检查是否可以在symbol1买入，在symbol2卖出
        if best_ask1 < best_bid2 - self.arbitrage_threshold:
            print(f"套利机会: {symbol1} -> {symbol2}, 价差: {best_bid2 - best_ask1}")
            # 买入symbol1
            self.submit_order(
                symbol=symbol1,
                side='buy',
                order_type='market',
                quantity=self.order_size
            )
            # 卖出symbol2
            self.submit_order(
                symbol=symbol2,
                side='sell',
                order_type='market',
                quantity=self.order_size
            )
        
        # 检查是否可以在symbol2买入，在symbol1卖出
        if best_ask2 < best_bid1 - self.arbitrage_threshold:
            print(f"套利机会: {symbol2} -> {symbol1}, 价差: {best_bid1 - best_ask2}")
            # 买入symbol2
            self.submit_order(
                symbol=symbol2,
                side='buy',
                order_type='market',
                quantity=self.order_size
            )
            # 卖出symbol1
            self.submit_order(
                symbol=symbol1,
                side='sell',
                order_type='market',
                quantity=self.order_size
            )

class MomentumStrategy(BaseStrategy):
    """动量策略：基于价格趋势交易"""
    
    def __init__(self, strategy_id: str, exchange: ExchangeSimulator, symbols: List[str]):
        super().__init__(strategy_id, exchange, symbols)
        self.price_history: Dict[str, Deque[float]] = {
            symbol: deque(maxlen=50) for symbol in symbols
        }  # 价格历史
        self.trend_threshold = 0.01  # 趋势阈值
        self.order_size = 8  # 订单数量
        self.update_frequency = 0.02  # 策略更新频率
    
    def on_market_data_update(self, symbol: str, order_book: OrderBook) -> None:
        """市场数据更新时记录价格"""
        mid_price = (order_book.get_best_bid() + order_book.get_best_ask()) / 2 if \
                   order_book.get_best_bid() and order_book.get_best_ask() else None
                   
        if mid_price:
            self.price_history[symbol].append(mid_price)
    
    def run(self) -> None:
        """策略主循环"""
        while self.running:
            for symbol in self.symbols:
                self.check_momentum(symbol)
            time.sleep(self.update_frequency)
    
    def check_momentum(self, symbol: str) -> None:
        """检查价格动量"""
        prices = self.price_history[symbol]
        if len(prices) < 20:  # 需要足够的历史数据
            return
            
        # 计算短期和长期移动平均线
        short_ma = np.mean(list(prices)[-5:])  # 5期均线
        long_ma = np.mean(list(prices)[-20:])  # 20期均线
        
        # 金叉：短期均线上穿长期均线，买入信号
        if short_ma > long_ma * (1 + self.trend_threshold):
            print(f"买入信号: {symbol}, 短期均线: {short_ma}, 长期均线: {long_ma}")
            self.submit_order(
                symbol=symbol,
                side='buy',
                order_type='market',
                quantity=self.order_size
            )
        
        # 死叉：短期均线下穿长期均线，卖出信号
        elif short_ma < long_ma * (1 - self.trend_threshold):
            print(f"卖出信号: {symbol}, 短期均线: {short_ma}, 长期均线: {long_ma}")
            self.submit_order(
                symbol=symbol,
                side='sell',
                order_type='market',
                quantity=self.order_size
            )

# ------------------------------
# 市场数据模拟器
# ------------------------------

class MarketDataSimulator:
    """模拟市场数据生成"""
    
    def __init__(self, exchange: ExchangeSimulator, symbols: List[str]):
        self.exchange = exchange
        self.symbols = symbols
        self.prices: Dict[str, float] = {symbol: 100.0 + random.uniform(-10, 10) for symbol in symbols}
        self.volatility: Dict[str, float] = {symbol: random.uniform(0.01, 0.05) for symbol in symbols}
        self.running = False
        self.thread = None
        self.update_frequency = 0.005  # 5ms更新一次，模拟高频数据
    
    def start(self) -> None:
        """开始生成市场数据"""
        self.running = True
        self.thread = threading.Thread(target=self.generate_data, daemon=True)
        self.thread.start()
    
    def stop(self) -> None:
        """停止生成市场数据"""
        self.running = False
        if self.thread:
            self.thread.join()
    
    def generate_data(self) -> None:
        """生成模拟市场数据"""
        while self.running:
            for symbol in self.symbols:
                # 基于几何布朗运动模型生成价格
                dt = self.update_frequency
                drift = 0.001 * dt
                shock = random.gauss(0, self.volatility[symbol] * np.sqrt(dt))
                self.prices[symbol] *= np.exp(drift + shock)
                self.prices[symbol] = round(self.prices[symbol], 4)
                
                # 生成买卖价差
                spread = random.uniform(0.005, 0.03)
                bid_price = self.prices[symbol]
                ask_price = bid_price + spread
                
                # 生成买卖量
                bid_size = random.randint(50, 500)
                ask_size = random.randint(50, 500)
                
                # 更新订单簿
                order_book = self.exchange.get_order_book(symbol)
                order_book.update_level('buy', bid_price, bid_size)
                order_book.update_level('sell', ask_price, ask_size)
                
                # 随机生成一些市场订单，增加流动性
                if random.random() < 0.3:  # 30%概率生成订单
                    side = 'buy' if random.random() < 0.5 else 'sell'
                    quantity = random.randint(1, 20)
                    self.exchange.order_manager.submit_order(
                        symbol=symbol,
                        side=side,
                        order_type='market',
                        quantity=quantity,
                        strategy_id='simulator'
                    )
            
            # 控制更新频率
            time.sleep(self.update_frequency)

# ------------------------------
# 主程序
# ------------------------------

if __name__ == "__main__":
    print("高级高频交易模拟系统启动（仅用于演示）")
    
    # 初始化交易所
    exchange = ExchangeSimulator()
    
    # 定义交易品种
    symbols = ["AAPL", "MSFT", "GOOG", "AMZN", "TSLA"]
    symbol_pairs = [("AAPL", "MSFT"), ("GOOG", "AMZN")]  # 假设这些交易对存在套利关系
    
    # 启动市场数据模拟器
    data_simulator = MarketDataSimulator(exchange, symbols)
    data_simulator.start()
    
    # 初始化并启动多种策略
    strategies = [
        MarketMakingStrategy("market_maker_1", exchange, symbols),
        ArbitrageStrategy("arbitrage_1", exchange, symbol_pairs),
        MomentumStrategy("momentum_1", exchange, symbols)
    ]
    
    for strategy in strategies:
        strategy.start()
        print(f"策略 {strategy.strategy_id} 已启动")
    
    try:
        # 运行10秒后停止
        simulation_duration = 10
        print(f"模拟将运行 {simulation_duration} 秒...")
        time.sleep(simulation_duration)
    finally:
        # 停止所有组件
        for strategy in strategies:
            strategy.stop()
            print(f"策略 {strategy.strategy_id} 已停止")
        
        data_simulator.stop()
        print("市场数据模拟器已停止")
        
        # 打印最终风险指标
        print("\n最终风险指标:")
        for symbol in symbols:
            metrics = exchange.risk_manager.get_risk_metrics(symbol)
            if metrics:
                print(f"{symbol}: 持仓={metrics.current_position}, 当日盈亏={metrics.daily_pnl:.2f}, 最大回撤={metrics.max_drawdown:.2f}")
        
        print("模拟结束")
    

Exception in thread Thread-5:
Traceback (most recent call last):
  File "D:\anaconda\envs\financial-ml\lib\threading.py", line 980, in _bootstrap_inner
Exception in thread Thread-10:
Traceback (most recent call last):
  File "D:\anaconda\envs\financial-ml\lib\threading.py", line 980, in _bootstrap_inner
Exception in thread Thread-15:
Traceback (most recent call last):
  File "D:\anaconda\envs\financial-ml\lib\threading.py", line 980, in _bootstrap_inner
    self.run()
  File "D:\anaconda\envs\financial-ml\lib\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
Exception in thread Thread-17:
Traceback (most recent call last):
  File "D:\anaconda\envs\financial-ml\lib\threading.py", line 980, in _bootstrap_inner
    self.run()
  File "D:\anaconda\envs\financial-ml\lib\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
    self.run()
  File "D:\anaconda\envs\financial-ml\lib\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
    _threading_Thread_run(sel

高级高频交易模拟系统启动（仅用于演示）
策略 market_maker_1 已启动
套利机会: MSFT -> AAPL, 价差: 8.837384025067962
策略 arbitrage_1 已启动
策略 momentum_1 已启动
模拟将运行 10 秒...
套利机会: GOOG -> AMZN, 价差: 2.3928643511042083
套利机会: MSFT -> AAPL, 价差: 8.838384025067967
套利机会: GOOG -> AMZN, 价差: 2.6258203850837845
套利机会: MSFT -> AAPL, 价差: 9.16692439504331
套利机会: GOOG -> AMZN, 价差: 2.659323093145929
套利机会: MSFT -> AAPL, 价差: 9.400424395043302
套利机会: GOOG -> AMZN, 价差: 2.791044759589127
套利机会: MSFT -> AAPL, 价差: 9.603706531088662
套利机会: GOOG -> AMZN, 价差: 3.24344400330493
套利机会: MSFT -> AAPL, 价差: 9.68474680167374
套利机会: GOOG -> AMZN, 价差: 3.7263971088088965
套利机会: MSFT -> AAPL, 价差: 9.953546801673738
套利机会: GOOG -> AMZN, 价差: 3.958497108808899
套利机会: MSFT -> AAPL, 价差: 9.953546801673738
套利机会: GOOG -> AMZN, 价差: 3.988239040181
套利机会: MSFT -> AAPL, 价差: 10.219669794970883
套利机会: GOOG -> AMZN, 价差: 4.253601932337332
套利机会: MSFT -> AAPL, 价差: 10.309369794970891
套利机会: GOOG -> AMZN, 价差: 4.695255689684927


    _threading_Thread_run(self)
  File "D:\anaconda\envs\financial-ml\lib\threading.py", line 917, in run
    self.run()
  File "D:\anaconda\envs\financial-ml\lib\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
    self.run()
  File "D:\anaconda\envs\financial-ml\lib\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
Exception in thread Thread-78:
Traceback (most recent call last):
  File "D:\anaconda\envs\financial-ml\lib\threading.py", line 980, in _bootstrap_inner
    self._target(*self._args, **self._kwargs)
  File "C:\Users\86157\AppData\Local\Temp\ipykernel_3880\4214617836.py", line 326, in process_order
    self.run()
  File "D:\anaconda\envs\financial-ml\lib\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
StopIteration
    self.run()
  File "D:\anaconda\envs\financial-ml\lib\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
  File "C:\Users\86157\AppData\Local\Temp\ipykernel_3880\4214617836.py", line 351, in process_market_

套利机会: MSFT -> AAPL, 价差: 10.38598453603855
套利机会: GOOG -> AMZN, 价差: 5.014825540441322
套利机会: MSFT -> AAPL, 价差: 10.38598453603855
套利机会: GOOG -> AMZN, 价差: 5.014825540441322
套利机会: MSFT -> AAPL, 价差: 10.38598453603855
套利机会: GOOG -> AMZN, 价差: 5.014825540441322
套利机会: MSFT -> AAPL, 价差: 10.676469986938145
套利机会: GOOG -> AMZN, 价差: 5.152313128269796
套利机会: MSFT -> AAPL, 价差: 11.077690884527968
套利机会: GOOG -> AMZN, 价差: 5.573288348458263
套利机会: MSFT -> AAPL, 价差: 11.340790884527962
套利机会: GOOG -> AMZN, 价差: 5.573288348458263
套利机会: MSFT -> AAPL, 价差: 11.838190884527961
套利机会: GOOG -> AMZN, 价差: 5.573288348458263
套利机会: MSFT -> AAPL, 价差: 11.838190884527961
套利机会: GOOG -> AMZN, 价差: 5.573288348458263
套利机会: MSFT -> AAPL, 价差: 11.982790884527958
套利机会: GOOG -> AMZN, 价差: 6.404688348458265
套利机会: MSFT -> AAPL, 价差: 11.982790884527958
套利机会: GOOG -> AMZN, 价差: 6.480888348458265
套利机会: MSFT -> AAPL, 价差: 12.142290884527966
套利机会: GOOG -> AMZN, 价差: 6.480888348458265
套利机会: MSFT -> AAPL, 价差: 12.142290884527966
套利机会: GOOG -> AMZN, 价差: 6

    StopIteration
  File "C:\Users\86157\AppData\Local\Temp\ipykernel_3880\4214617836.py", line 351, in process_market_order
_threading_Thread_run(self)
  File "D:\anaconda\envs\financial-ml\lib\threading.py", line 917, in run
Exception in thread Thread-195:
Traceback (most recent call last):
  File "D:\anaconda\envs\financial-ml\lib\threading.py", line 980, in _bootstrap_inner
  File "C:\Users\86157\AppData\Local\Temp\ipykernel_3880\4214617836.py", line 351, in process_market_order
    _threading_Thread_run(self)
  File "D:\anaconda\envs\financial-ml\lib\threading.py", line 917, in run
    self.run()
  File "D:\anaconda\envs\financial-ml\lib\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
Exception in thread Thread-196:
Traceback (most recent call last):
  File "D:\anaconda\envs\financial-ml\lib\threading.py", line 980, in _bootstrap_inner
StopIteration
Exception in thread Thread-197:
Traceback (most recent call last):
  File "D:\anaconda\envs\financial-ml\lib\threading

套利机会: MSFT -> AAPL, 价差: 12.762590884527967
套利机会: GOOG -> AMZN, 价差: 6.746488348458257
套利机会: MSFT -> AAPL, 价差: 12.872490884527963
套利机会: GOOG -> AMZN, 价差: 6.746488348458257
套利机会: MSFT -> AAPL, 价差: 12.902490884527964
套利机会: GOOG -> AMZN, 价差: 6.746488348458257
套利机会: MSFT -> AAPL, 价差: 12.902490884527964
套利机会: GOOG -> AMZN, 价差: 6.746488348458257
套利机会: MSFT -> AAPL, 价差: 12.902490884527964
套利机会: GOOG -> AMZN, 价差: 6.746488348458257
套利机会: MSFT -> AAPL, 价差: 12.902490884527964
套利机会: GOOG -> AMZN, 价差: 6.746488348458257
套利机会: MSFT -> AAPL, 价差: 13.228490884527957
套利机会: GOOG -> AMZN, 价差: 6.746488348458257
套利机会: MSFT -> AAPL, 价差: 13.228490884527957
套利机会: GOOG -> AMZN, 价差: 6.746488348458257
套利机会: MSFT -> AAPL, 价差: 13.382590884527957
套利机会: GOOG -> AMZN, 价差: 6.746488348458257
套利机会: MSFT -> AAPL, 价差: 13.422990884527962
套利机会: GOOG -> AMZN, 价差: 6.746488348458257
套利机会: MSFT -> AAPL, 价差: 13.430790884527966
套利机会: GOOG -> AMZN, 价差: 6.746488348458257
套利机会: MSFT -> AAPL, 价差: 13.430790884527966
套利机会: GOOG -> AMZN, 价差

    self.run()
  File "D:\anaconda\envs\financial-ml\lib\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
StopIteration
StopIteration
Exception in thread Thread-635:
Traceback (most recent call last):
  File "D:\anaconda\envs\financial-ml\lib\threading.py", line 980, in _bootstrap_inner
    self._target(*self._args, **self._kwargs)
  File "C:\Users\86157\AppData\Local\Temp\ipykernel_3880\4214617836.py", line 326, in process_order
  File "C:\Users\86157\AppData\Local\Temp\ipykernel_3880\4214617836.py", line 351, in process_market_order
    self._target(*self._args, **self._kwargs)
  File "C:\Users\86157\AppData\Local\Temp\ipykernel_3880\4214617836.py", line 326, in process_order
    self._target(*self._args, **self._kwargs)
  File "C:\Users\86157\AppData\Local\Temp\ipykernel_3880\4214617836.py", line 326, in process_order
    self.run()
  File "D:\anaconda\envs\financial-ml\lib\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
  File "C:\Users\86157\AppData\Lo